# Robust04 Final Project (Part A): Retrieval Runs + Reproducibility

## Goal

Build and submit **three different retrieval runs** on the Robust04 collection (via Pyserini), optimizing **MAP**.

## Dataset split (given by the course)

- **Training/tuning**: first 50 queries in `Files-20260104/queriesROBUST.txt` (with judgments in `Files-20260104/qrels_50_Queries`)
- **Test/submission**: remaining 199 queries (no judgments)

## What this notebook does

1. **Loads queries + qrels** and evaluates methods on the 50 judged queries (MAP).
2. Reproduces the three submitted runs for the 199 test queries:

- **`run_1.res`**: BM25 + RM3 (pseudo-relevance feedback)
- **`run_2.res`**: entity-lite query expansion (capitalized phrase feedback from top RM3 docs) + second-pass BM25
- **`run_3.res`**: score fusion (RM3 + SPLADE++ + SPLADE-v3 + Dense) **+ MonoT5 passage-level reranking of top-200**

## Key references (brief)

- BM25: Robertson and Zaragoza, 2009
- RM3 PRF: Lavrenko and Croft, 2001
- SPLADE: Formal et al., 2021
- BGE embeddings: BAAI (BGE family)
- MonoT5 reranking: Nogueira et al., 2020; Nogueira and Lin, 2019
- Passage-level aggregation (PARADE): Li et al., 2020


In [ ]:
import os
from collections import defaultdict
from pathlib import Path
import subprocess
import sys
import zipfile

# Prevent Lucene memory-segment issues in some environments
os.environ.setdefault(
    "JAVA_TOOL_OPTIONS",
    "-Dorg.apache.lucene.store.MMapDirectory.enableMemorySegments=false",
)

import torch

from pyserini.encode import SpladeQueryEncoder
from pyserini.search.lucene import LuceneHnswDenseSearcher, LuceneImpactSearcher, LuceneSearcher

print("torch:", torch.__version__)
print("device:", "cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
QUERIES_PATH = Path('Files-20260104/queriesROBUST.txt')
QRELS_PATH = Path('Files-20260104/qrels_50_Queries')


def read_queries_tsv(path: Path):
    queries = {}
    for line in path.read_text(encoding='utf-8').splitlines():
        line = line.strip()
        if not line:
            continue
        qid, query = line.split('\t', 1)
        queries[qid] = query
    return queries


def read_qrels(path: Path):
    qrels = defaultdict(dict)
    for line in path.read_text(encoding='utf-8').splitlines():
        parts = line.strip().split()
        if len(parts) != 4:
            continue
        qid, _, docid, rel = parts
        qrels[qid][docid] = int(rel)
    return qrels


def average_precision(docids, rels):
    num_rel = sum(1 for r in rels.values() if r > 0)
    if num_rel == 0:
        return 0.0
    hit = 0
    s = 0.0
    for i, d in enumerate(docids, start=1):
        if rels.get(d, 0) > 0:
            hit += 1
            s += hit / i
    return s / num_rel


def mean_ap(run, qrels):
    return sum(average_precision(run[qid], qrels[qid]) for qid in run) / len(run)


def minmax_norm(scores_dict):
    if not scores_dict:
        return {}
    vals = list(scores_dict.values())
    mn, mx = min(vals), max(vals)
    if mx - mn < 1e-9:
        return {d: 0.0 for d in scores_dict}
    return {d: (s - mn) / (mx - mn) for d, s in scores_dict.items()}


def fuse_weighted_minmax(runs_scores, weights, depth=1000):
    """Min-max normalize each run's scores, then weighted-sum fuse."""
    norms = [minmax_norm(rs) for rs in runs_scores]
    docs = set()
    for n in norms:
        docs |= set(n.keys())
    fused_scores = {}
    for d in docs:
        s = 0.0
        for w, n in zip(weights, norms):
            s += w * n.get(d, 0.0)
        fused_scores[d] = s
    ranked = sorted(fused_scores.items(), key=lambda x: (-x[1], x[0]))
    return [d for d, _ in ranked[:depth]]


def retrieve_run(searcher, queries, k=1000):
    run = {}
    for qid, query in queries.items():
        hits = searcher.search(query, k=k)
        run[qid] = [h.docid for h in hits]
    return run


def retrieve_scores(searcher, queries, k=1000):
    scores = {}
    for qid, query in queries.items():
        hits = searcher.search(query, k=k)
        scores[qid] = {h.docid: float(h.score) for h in hits}
    return scores


all_queries = read_queries_tsv(QUERIES_PATH)
train_qids = list(all_queries.keys())[:50]
train_queries = {qid: all_queries[qid] for qid in train_qids}
qrels = read_qrels(QRELS_PATH)

print('loaded queries:', len(all_queries))
print('train queries:', len(train_queries))
print('qrels qids:', len(qrels))

device = 'cuda' if torch.cuda.is_available() else 'cpu'
device

In [ ]:
# 1) Methods on the 50 judged queries (MAP)

# Method 1: BM25 + RM3
rm3 = LuceneSearcher.from_prebuilt_index('robust04')
rm3.set_bm25(0.9, 0.4)
rm3.set_rm3(20, 5, 0.5)

# Method 2/3: learned sparse retrieval (SPLADE family)
spladepp_encoder = SpladeQueryEncoder('naver/splade-cocondenser-ensembledistil', device=device)
spladepp = LuceneImpactSearcher.from_prebuilt_index('beir-v1.0.0-robust04.splade-pp-ed', spladepp_encoder)

spladev3_encoder = SpladeQueryEncoder('naver/splade-v3-distilbert', device=device)
spladev3 = LuceneImpactSearcher.from_prebuilt_index('beir-v1.0.0-robust04.splade-v3', spladev3_encoder)

# Dense retrieval (BGE embeddings + Lucene HNSW)
dense = LuceneHnswDenseSearcher.from_prebuilt_index(
    'beir-v1.0.0-robust04.bge-base-en-v1.5.hnsw',
    ef_search=1000,
    encoder='BgeBaseEn15',
)

rm3_run = retrieve_run(rm3, train_queries)
spladepp_run = retrieve_run(spladepp, train_queries)
spladev3_run = retrieve_run(spladev3, train_queries)
dense_run = retrieve_run(dense, train_queries)

print('RM3 MAP:', f'{mean_ap(rm3_run, qrels):.4f}')
print('SPLADE++ MAP:', f'{mean_ap(spladepp_run, qrels):.4f}')
print('SPLADE-v3-distil MAP:', f'{mean_ap(spladev3_run, qrels):.4f}')
print('Dense (BGE) MAP:', f'{mean_ap(dense_run, qrels):.4f}')

rm3_scores = retrieve_scores(rm3, train_queries)
spladepp_scores = retrieve_scores(spladepp, train_queries)
spladev3_scores = retrieve_scores(spladev3, train_queries)
dense_scores = retrieve_scores(dense, train_queries)

# Fused methods (min-max normalization + weighted sum)
# Note: in the final submission, run_2 is entity-lite query expansion; the fusion results here are baselines.
fusion_3way_weights = (0.60, 0.25, 0.15)          # rm3, splade++, dense
fusion_4way_weights = (0.55, 0.10, 0.15, 0.20)    # rm3, splade++, splade-v3, dense

fusion_3way = {
    qid: fuse_weighted_minmax([rm3_scores[qid], spladepp_scores[qid], dense_scores[qid]], fusion_3way_weights)
    for qid in train_queries
}
fusion_4way = {
    qid: fuse_weighted_minmax([rm3_scores[qid], spladepp_scores[qid], spladev3_scores[qid], dense_scores[qid]], fusion_4way_weights)
    for qid in train_queries
}

print('Fusion (RM3+SPLADE+++Dense) MAP:', f'{mean_ap(fusion_3way, qrels):.4f}')
print('Fusion (RM3+SPLADE+++SPLADE-v3+Dense) MAP:', f'{mean_ap(fusion_4way, qrels):.4f}')

# Always close searchers to avoid JVM/mmap resource issues
rm3.close(); spladepp.close(); spladev3.close(); dense.close()

# 2) Generate submission runs for the 199 test queries

We generate three run files in **TREC 6-column format**:

- **`run_1.res`**: BM25+RM3 (lexical baseline)
- **`run_2.res`**: **Entity-lite query expansion**
  - Take top RM3 documents as pseudo-relevant feedback.
  - Extract **capitalized 1–3 word phrases** as a lightweight “entity” proxy.
  - Build a boosted Lucene query (original query terms + boosted phrase queries).
  - Run a **second-pass BM25** with this expanded query.
- **`run_3.res`**: fusion of RM3 + SPLADE++ + SPLADE-v3 + Dense **followed by MonoT5 passage-level reranking of the top-200**

To enable the entity-lite `run_2` in `generate_runs.py`, pass:

- `--run2-method entity_lite`

(There are additional knobs like `--run2-entity-fb-docs`, `--run2-entity-max-phrases`, and boosts/slop if you want to tune it.)

Passage-level MonoT5 settings (tuned on the 50 judged queries):

- `model = zeta-alpha-ai/monot5-3b-inpars-v2-robust04`
- `top_n = 200`
- `alpha = 0.2` where `alpha` controls interpolation between fused score and reranker score:
  - `combined = alpha * fused + (1-alpha) * reranker`
- Passage splitting + aggregation:
  - `doc_max_chars = 12000`
  - `passage_chars = 1500`
  - `stride_chars = 1200`
  - `max_passages = 8`
  - aggregation = `max` (MaxP)

Note: `generate_runs.py` supports additional passage aggregation modes via `--monot5p-agg`:

- `max` (MaxP)
- `avg_topk`
- `softmax`
- `hybrid` (interpolates MaxP and AvgTopK; controlled by `--monot5p-hybrid-lambda` and `--monot5p-avg-topk`)

Note: passage-level reranking helps on Robust04 because documents can be long; a single 512-token truncation can miss the relevant part.


In [ ]:
subprocess.run(
    [
        sys.executable,
        'generate_runs.py',
        '--out1', 'run_1.res',
        '--out2', 'run_2.res',
        '--out3', 'run_3.res',
        '--run2-method', 'entity_lite',
        '--rerank3-monot5-passages',
        '--monot5p-model', 'zeta-alpha-ai/monot5-3b-inpars-v2-robust04',
        '--monot5p-alpha', '0.2',
        '--monot5p-top-n', '200',
        '--monot5p-batch-size', '4',
        '--monot5p-max-length', '512',
        '--monot5p-doc-max-chars', '12000',
        '--monot5p-passage-chars', '1500',
        '--monot5p-stride-chars', '1200',
        '--monot5p-max-passages', '8',
        '--monot5p-agg', 'max',
        '--monot5p-fp16',
    ],
    check=True,
)

In [ ]:
# 3) Sanity-check the generated run files

EXPECTED_QIDS = set(map(str, list(range(351, 451)) + list(range(601, 672)) + list(range(673, 701))))


def sanity_check_run(path):
    qids = set()
    bad = 0
    line_count = 0
    last_qid = None
    last_rank = 0
    last_score = None
    with Path(path).open('r', encoding='utf-8') as f:
        for line in f:
            line_count += 1
            parts = line.strip().split()
            if len(parts) != 6:
                bad += 1
                continue
            qid, q0, docid, rank, score, tag = parts
            qids.add(qid)
            if q0 != 'Q0':
                bad += 1
            r = int(rank)
            s = float(score)
            if last_qid != qid:
                last_qid = qid
                last_rank = 0
                last_score = None
            if r != last_rank + 1:
                bad += 1
            if last_score is not None and s > last_score + 1e-6:
                bad += 1
            last_rank = r
            last_score = s
    missing = EXPECTED_QIDS - qids
    extra = qids - EXPECTED_QIDS
    return {
        'lines': line_count,
        'unique_qids': len(qids),
        'missing_qids': len(missing),
        'extra_qids': len(extra),
        'bad_checks': bad,
    }

for fname in ['run_1.res', 'run_2.res', 'run_3.res']:
    print(fname, sanity_check_run(fname))

In [ ]:
# 4) Package runs into a single ZIP file

zip_name = 'Final_Project_Part_A_runs.zip'
with zipfile.ZipFile(zip_name, 'w', compression=zipfile.ZIP_DEFLATED) as z:
    for fname in ['run_1.res', 'run_2.res', 'run_3.res']:
        z.write(fname, arcname=fname)
zip_name